## Publish a Line-segments object

This example shows how to convert line-segments data  in CSV format into an Evo geoscience object using the Evo Python SDK.

### Requirements

You must have a Seequent account with the Evo entitlement to use this notebook.

The following parameters must be provided:

- The client ID of your Evo application.
- The callback/redirect URL of your Evo application.

To obtain these app credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens) in the Seequent Developer Portal.

In [ ]:
import os

# Evo app credentials (uses environment variables in CI, or replace with your own)
client_id = os.environ.get("EVO_CLIENT_ID", "<your-client-id>")
redirect_url = os.environ.get("EVO_REDIRECT_URL", "<your-redirect-url>")

# CI authentication (uses service app credentials)
if os.environ.get("CI") and os.environ.get("EVO_CLIENT_SECRET"):
    import sys

    sys.path.insert(0, "../..")
    from auth_helper import _create_ci_manager

    manager = await _create_ci_manager()
else:
    # Interactive authentication
    from evo.notebooks import ServiceManagerWidget

    manager = await ServiceManagerWidget.with_auth_code(
        client_id=client_id,
        redirect_url=redirect_url,
    ).login()

### Use the Evo Python SDK to create an object client and a data client

In [ ]:
from evo.objects import ObjectAPIClient

# The object client will manage your auth token and Geoscience Object API requests.
object_client = ObjectAPIClient(manager.get_environment(), manager.get_connector())

# The data client will manage saving your data as Parquet and publishing your data to Evo storage.
data_client = object_client.get_data_client(manager.cache)

### Define helper functions

These functions assist with assembling the elements and components of geoscience objects and for viewing the new object in the Evo portal.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display


def build_vertices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract X, Y, Z coordinates as vertices.
    Args:
        df: DataFrame with columns ['pid', 'X', 'Y', 'Z']
    Returns:
        DataFrame with 3 columns: X, Y, Z (float64)
    """
    return df[["X", "Y", "Z"]].astype("float64")


def build_indices_per_pid(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build segment indices WITHIN each pid(polyline separator) (not global continuity).
    Each segment connects two consecutive points in the same pid.
    Format: (n0, n1) where n0 and n1 are vertex indices.
    Args:
        df: DataFrame with columns ['pid', 'X', 'Y', 'Z']
    Returns:
        DataFrame with 2 columns: n0, n1 (uint64)
    """
    indices = []
    offset = 0  # absolute vertex index counter

    for pid in df["pid"].unique():
        pid_rows = df[df["pid"] == pid]
        num_points = len(pid_rows)

        if num_points == 1:
            indices.append({"n0": offset, "n1": offset})
        else:
            for i in range(num_points - 1):
                indices.append({"n0": offset + i, "n1": offset + i + 1})

        offset += num_points

    return pd.DataFrame(indices).astype({"n0": "uint64", "n1": "uint64"})


def build_chunks(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build chunk metadata for each pid.
    Chunks define where each pid's segments start and how many segments it has.
    Args:
        df: DataFrame with columns ['pid', 'X', 'Y', 'Z']
    Returns:
        DataFrame with columns: pid, start_segment_index, number_of_segment

    """
    chunks = df.groupby("pid", sort=False).size().sub(1).reset_index(name="number_of_segment")

    chunks["number_of_segment"] = chunks["number_of_segment"].clip(lower=1)

    chunks["start_segment_index"] = chunks["number_of_segment"].shift(fill_value=0).cumsum()

    return chunks[["pid", "start_segment_index", "number_of_segment"]].astype(
        {
            "start_segment_index": "uint64",
            "number_of_segment": "uint64",
        }
    )


def create_category_lookup_and_values(attribute):
    """
    Create a category lookup table and the associated column of mapped key values.
    Args:
        attribute (pd.DataFrame): An attribute of a geoscience object.
    Returns:
        table_df (pd.DataFrame): The category lookup table.
        values_df (pd.DataFrame): The associated column with mapped key values.
    """
    # Replace NaN with empty string
    attribute.replace(np.nan, "", regex=True, inplace=True)
    set_obj = set(attribute["data"])

    list_obj = list(set_obj)
    list_obj.sort()
    num_unique_elements = len(list_obj)

    # Create lookup table
    table_df = pd.DataFrame([])
    table_df["key"] = pd.Series(range(1, num_unique_elements + 1), dtype="int64")
    table_df["value"] = list_obj

    # Create data column
    values_df = pd.DataFrame([])
    values_df["data"] = attribute["data"].map(table_df.set_index("value")["key"]).astype("Int64")
    return table_df, values_df


def build_portal_url(object_metadata):
    """
    Build and display a link to view the geoscience object in the Evo Portal.
    Args:
        object_metadata: The metadata object returned after creating the geoscience object.
    Returns:
        None. Displays an HTML link to the Evo Portal for the created object.
    """

    hub_url = object_metadata.environment.hub_url
    hub_name = hub_url.split("://")[1].split(".")[0]
    org_id = object_metadata.environment.org_id
    workspace_id = object_metadata.environment.workspace_id
    object_id = object_metadata.id

    url = f"https://evo.seequent.com/{org_id}/workspaces/{hub_name}/{workspace_id}/overview?id={object_id}"

    display(HTML(f'<a href="{url}" target="_blank">View line segment object in the Evo Portal</a>'))

### Define object metadata

Geoscience object data must conform to a specific object schema. The `evo-schemas` package provides Pydantic models that make it easy to work with the equivalent JSON schemas. 
For this example we'll use v2.2.0 of the line-segments schema, via the relevant Pydantic model.

Enter values for these parameters that are required by the object schema.
- `object_name`: The name of the object.
- `object_path`: The file path where the object will be found.
- `object_epsg_code`: (Optional) The EPSG region code that matches the location of your data. Leave as `None` if not required.
- `object_tags`: (Optional) A dictionary of additional tags to be assigned to the object. Leave as `None` is not required.

In [ ]:
import os

from evo_schemas.components import Crs_V1_0_1_EpsgCode

input_path = "sample-data"

object_name = "LineSegment_SDK_demo"
if os.environ.get("CI"):
    from datetime import datetime, timezone

    object_name += f"_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
object_path = "Jupyter_Example"
object_tags = {"Source": "Jupyter Notebook"}

# Define the coordinate reference system (CRS) to be unspecified.
# coordinate_reference_system = "unspecified"
object_epsg_code = 32650

# Define a coordinate reference system (CRS) for the object.
coordinate_reference_system = Crs_V1_0_1_EpsgCode(epsg_code=object_epsg_code)

# Define input and output file paths.
input_file = f"{input_path}/SP_Mapping.csv"

# Load the input csv file.
input_df = pd.read_csv(input_file)

# Define the object path.
full_obj_path = f"{object_path}/{object_name}.json"

### Define object attributes and keys

In [ ]:
import uuid

# List all of the attributes to be included in the object. Every attribute must have a unique key associated with it.
# Keys must be unique across the entire object, and we recommend saving a reference to the keys for later use.
object_attributes = {
    "LS_data": {
        "pid": str(uuid.uuid4()),
        "X": str(uuid.uuid4()),
        "Y": str(uuid.uuid4()),
        "Z": str(uuid.uuid4()),
        "Type": str(uuid.uuid4()),
        "Comment": str(uuid.uuid4()),
        "Digitised_by": str(uuid.uuid4()),
    },
}

### Coordinates

In [ ]:
from evo_schemas.components import BoundingBox_V1_0_1

# Create a dataframe for the coordinates.
coordinates_df = input_df[["X", "Y", "Z"]]

# Create a bounding box for the coordinates.
bounding_box = BoundingBox_V1_0_1(
    min_x=coordinates_df["X"].min(),
    max_x=coordinates_df["X"].max(),
    min_y=coordinates_df["Y"].min(),
    max_y=coordinates_df["Y"].max(),
    min_z=coordinates_df["Z"].min(),
    max_z=coordinates_df["Z"].max(),
)

### Creating attributes

In [ ]:
from evo_schemas.components import (
    CategoryAttribute_V1_1_0,
    ContinuousAttribute_V1_1_0,
    NanCategorical_V1_0_1,
    NanContinuous_V1_0_1,
    StringAttribute_V1_1_0,
)
from evo_schemas.elements import (
    FloatArray1_V1_0_1,
    IntegerArray1_V1_0_1,
    LookupTable_V1_0_1,
    StringArray_V1_0_1,
)
from pandas.api.types import is_numeric_dtype

attribute_keys = object_attributes["LS_data"]
allowed_cols = [
    column_name
    for column_name in input_df.columns
    if column_name not in {"pid", "X", "Y", "Z"} and column_name in attribute_keys
]

chunks_df = build_chunks(input_df)
chunk_pid_order = chunks_df["pid"].tolist()
part_attribute_df = (
    input_df.groupby("pid", sort=False)[allowed_cols].first().reindex(chunk_pid_order).reset_index(drop=True)
)

lookup_list_columns = {"Type", "Digitised_by"}
string_columns = {"Comment"}

attributes = []

for heading_name, heading_key in attribute_keys.items():
    if heading_name not in allowed_cols:
        continue

    series = part_attribute_df[heading_name]
    values_df = pd.DataFrame({"data": series})
    print(f"Processing {heading_name}: {len(values_df)} rows")

    if heading_name in lookup_list_columns:
        table_df, mapped_values_df = create_category_lookup_and_values(values_df.copy())

        table = LookupTable_V1_0_1.from_dict(data_client.save_dataframe(table_df))
        values = IntegerArray1_V1_0_1.from_dict(data_client.save_dataframe(mapped_values_df.astype({"data": "int32"})))

        attribute = CategoryAttribute_V1_1_0(
            name=heading_name,
            nan_description=NanCategorical_V1_0_1(values=[]),
            key=heading_key,
            table=table,
            values=values,
        )
        attributes.append(attribute)

    elif heading_name in string_columns:
        values_df["data"] = values_df.astype(str).reset_index(drop=True)
        values = StringArray_V1_0_1.from_dict(data_client.save_dataframe(values_df))
        attribute = StringAttribute_V1_1_0(
            name=heading_name,
            key=heading_key,
            values=values,
        )
        attributes.append(attribute)

    else:
        non_null = series.dropna()
        is_numeric_column = is_numeric_dtype(series) or (
            not non_null.empty and pd.to_numeric(non_null, errors="coerce").notna().all()
        )

        if not is_numeric_column:
            print(f"Skipping unsupported attribute {heading_name}: expected numeric/string/lookup")
            continue

        values_df["data"] = pd.to_numeric(series, errors="coerce").astype("float64")
        values = FloatArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))
        attribute = ContinuousAttribute_V1_1_0(
            name=heading_name,
            nan_description=NanContinuous_V1_0_1(values=[]),
            key=heading_key,
            values=values,
        )
        attributes.append(attribute)

### Build Segments and Parts

In [ ]:
from evo_schemas.components import Segments_V1_2_0, Segments_V1_2_0_Indices, Segments_V1_2_0_Vertices
from evo_schemas.elements import IndexArray2_V1_0_1
from evo_schemas.objects import LineSegments_V2_2_0_Parts

# Build geometry
vertices_df = build_vertices(input_df)
indices_df = build_indices_per_pid(input_df)

if len(indices_df) != int(chunks_df["number_of_segment"].sum()):
    raise ValueError(
        f"Indices/chunks mismatch: indices={len(indices_df)} vs sum(chunks)={int(chunks_df['number_of_segment'].sum())}"
    )

segments = Segments_V1_2_0(
    vertices=Segments_V1_2_0_Vertices.from_dict(data_client.save_dataframe(vertices_df)),
    indices=Segments_V1_2_0_Indices.from_dict(data_client.save_dataframe(indices_df)),
)

parts = LineSegments_V2_2_0_Parts(
    chunks=IndexArray2_V1_0_1.from_dict(
        data_client.save_dataframe(chunks_df[["start_segment_index", "number_of_segment"]])
    ),
    attributes=attributes,
)

### Create a new line segment and publish it to Evo

In [ ]:
# Lastly, assemble the complete geoscience object by combining all previously defined components.
# - The name and UUID are used to identify the object.
# - The UUID is set to None because this is a new object. A new UUID will be assigned by the Evo service.
# - The bounding box defines the spatial extent of the object.
# - The tags provide metadata about the object.
# - The coordinate reference system defines the spatial reference for the object.
# - The segments component contains the vertices and indices.
# - The parts component contains chunks and optional per-part attributes.

from evo_schemas.objects import LineSegments_V2_2_0

from evo.notebooks import FeedbackWidget

line_segments = LineSegments_V2_2_0(
    name=object_name,
    uuid=None,
    bounding_box=bounding_box,
    tags=object_tags,
    coordinate_reference_system=coordinate_reference_system,
    segments=segments,
    parts=parts,
)

# Upload the Parquet data to Evo.
await data_client.upload_referenced_data(line_segments.as_dict(), FeedbackWidget("Uploading data"))

# Create the geoscience object.
new_line_segments_metadata = await object_client.create_geoscience_object(full_obj_path, line_segments.as_dict())

### View the object in the Evo portal

In [ ]:
build_portal_url(new_line_segments_metadata)

Success! You now have a new geoscience object in Evo containing your line segment data.

## Summary

In this example, we've completed the following:
* Analysed the data columns and constructed the elements and components required for attribute.
* Converted the input data into vertices, indices, chunks and attribute data into Parquet format and saved it to the local cache.
* Combined all of the elements, components and data references into the line segment schema format.
* Uploaded the Parquet files and the newly assembled object in JSON format to Evo.